# Weight Histograms — Before & After Training

Weight distributions shift as the network learns.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
torch.manual_seed(42)
np.random.seed(42)

class SimpleMLP(nn.Module):
    def __init__(self, in_dim=8, hidden=16, n_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, n_classes),
        )
    def forward(self, x):
        return self.net(x)


## 1. Capture weights before training

In [ ]:
torch.manual_seed(42)
model = SimpleMLP(in_dim=8, hidden=32, n_classes=3)
before = {n: p.detach().clone() for n, p in model.named_parameters()}

## 2. Train briefly on synthetic data

In [ ]:
X = torch.randn(256, 8)
y = torch.randint(0, 3, (256,))
opt = torch.optim.Adam(model.parameters(), lr=0.01)
for _ in range(100):
    opt.zero_grad()
    F.cross_entropy(model(X), y).backward()
    opt.step()
after = {n: p.detach().clone() for n, p in model.named_parameters()}

## 3. Histogram: first layer weights before/after

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(before['net.0.weight'].flatten().numpy(), bins=40, color='steelblue', edgecolor='black')
axes[0].set_title('W1 before training')
axes[1].hist(after['net.0.weight'].flatten().numpy(), bins=40, color='coral', edgecolor='black')
axes[1].set_title('W1 after training')
plt.tight_layout(); plt.show()

## 4. Overlay all weight layers

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name in ['net.0.weight', 'net.2.weight']:
    axes[0].hist(before[name].flatten().numpy(), bins=30, alpha=0.6, label=name)
    axes[1].hist(after[name].flatten().numpy(), bins=30, alpha=0.6, label=name)
axes[0].legend(); axes[0].set_title('Before')
axes[1].legend(); axes[1].set_title('After')
plt.tight_layout(); plt.show()

## 5. Mean absolute weight per layer

In [ ]:
layers = list(before.keys())
means_before = [before[k].abs().mean().item() for k in layers]
means_after = [after[k].abs().mean().item() for k in layers]
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(layers)); w = 0.35
ax.bar(x - w/2, means_before, w, label='before', color='lightgray')
ax.bar(x + w/2, means_after, w, label='after', color='teal')
ax.set_xticks(x); ax.set_xticklabels([l.replace('.', '\n') for l in layers], fontsize=8)
ax.legend(); ax.set_title('Mean |weight| per parameter tensor')
plt.tight_layout(); plt.show()